# 02. 첫 번째 에이전트

**`create_agent()`로 시작하기**

## 학습 목표

LangChain v1의 `create_agent()`로 에이전트를 생성하고 실행합니다.

이 노트북을 마치면 다음을 할 수 있습니다:

- `@tool` 데코레이터로 커스텀 도구를 정의
- `create_agent()`로 에이전트를 생성
- `invoke()`로 에이전트를 실행하고 결과를 확인
- `stream()`으로 실시간 스트리밍 응답을 받기
- `InMemorySaver`로 멀티턴 대화를 구현

## 2.1 환경 설정

OpenAI를 통해 모델을 설정합니다. `ChatOpenAI`는 OpenAI 호환 API를 지원하므로, `base_url`을 변경하여 OpenAI를 사용할 수 있습니다.

In [1]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-4.1",
)
print("\u2713 모델 설정 완료:", model.model_name)

✓ 모델 설정 완료: gpt-4.1


In [2]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


Langfuse tracing ON — https://lf.ddok.ai


## 2.2 간단한 도구 만들기

`@tool` 데코레이터로 에이전트가 사용할 도구를 정의합니다.

도구를 정의할 때 주의할 점:
- **docstring**은 필수입니다. 에이전트가 도구의 용도를 이해하는 데 씁니다.
- **타입 힌트**를 쓰면 에이전트가 올바른 인자를 전달할 수 있습니다.
- 도구 이름은 함수명에서 자동으로 생성됩니다.

In [3]:
from langchain.tools import tool

@tool
def add(a: int, b: int) -> int:
    """두 수를 더합니다."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """두 수를 곱합니다."""
    return a * b

print("도구 목록:")
for t in [add, multiply]:
    print(f"  - {t.name}: {t.description}")

도구 목록:
  - add: 두 수를 더합니다.
  - multiply: 두 수를 곱합니다.


## 2.3 에이전트 생성

`create_agent()`로 모델과 도구를 결합합니다.

생성된 에이전트는 내부적으로 LangGraph 그래프로 구현되며, `invoke()`, `stream()` 등의 메서드를 제공합니다.

> LangChain v1에서는 `create_react_agent()` 대신 `create_agent()`를 사용합니다.

In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[add, multiply],
    system_prompt="당신은 수학 도우미입니다. 제공된 도구를 사용하여 계산하세요.",
)
print("\u2713 에이전트 생성 완료")
print(f"  타입: {type(agent).__name__}")

✓ 에이전트 생성 완료
  타입: CompiledStateGraph


## 2.4 에이전트 실행

`invoke()`로 에이전트를 실행합니다.

에이전트에 메시지를 전달하면, 내부적으로 ReAct 루프가 실행됩니다:
1. 모델이 질문을 분석하고 도구 호출을 결정
2. 도구가 실행되고 결과를 반환
3. 모델이 결과를 바탕으로 최종 응답을 생성

In [5]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "15 + 27은 얼마인가요?"}]},
    config=lf_config,
)

# 최종 응답 출력
print("에이전트 응답:")
print(result["messages"][-1].content)

에이전트 응답:
15 + 27은 42입니다.


In [6]:
# 전체 메시지 흐름 확인
print("전체 메시지 흐름:")
print("=" * 50)
for msg in result["messages"]:
    role = msg.type if hasattr(msg, 'type') else msg.get('role', 'unknown')
    content = msg.content if hasattr(msg, 'content') else msg.get('content', '')
    print(f"[{role}] {content[:200]}")
    print("-" * 50)

전체 메시지 흐름:
[human] 15 + 27은 얼마인가요?
--------------------------------------------------
[ai] 
--------------------------------------------------
[tool] 42
--------------------------------------------------
[ai] 15 + 27은 42입니다.
--------------------------------------------------


## 2.5 스트리밍 실행

`stream()`으로 실시간 응답을 받습니다.

스트리밍을 쓰면 에이전트의 각 단계(모델 추론, 도구 호출, 최종 응답)를 실시간으로 확인할 수 있습니다. `stream_mode="updates"`로 각 노드의 업데이트를 순차적으로 받을 수 있습니다.

In [7]:
print("스트리밍 실행:")
print("=" * 50)

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "12 * 8을 계산한 다음 결과에 5를 더하세요."}]},
    stream_mode="updates",
    config=lf_config,
):
    for node_name, update in chunk.items():
        print(f"\n--- {node_name} ---")
        if "messages" in update:
            for msg in update["messages"]:
                content = msg.content if hasattr(msg, 'content') else str(msg)
                if content:
                    print(f"  {content[:300]}")

스트리밍 실행:



--- model ---

--- tools ---
  96



--- model ---

--- tools ---
  101



--- model ---
  12 × 8 = 96이고, 여기에 5를 더하면 101입니다. 

최종 답: 101입니다.


## 2.6 멀티턴 대화

`InMemorySaver`로 대화 상태를 유지합니다.

`InMemorySaver`는 메모리 내에서 상태를 저장하며, `thread_id`로 대화 세션을 구분합니다.

> LangChain v1에서는 LangGraph의 체크포인터로 대화 히스토리를 관리합니다.

In [8]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

agent_with_memory = create_agent(
    model=model,
    tools=[add, multiply],
    system_prompt="당신은 수학 도우미입니다.",
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "math-session-1"}}

# 첫 번째 질문
result1 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "7 * 6은 얼마인가요?"}]},
    config={**config, **lf_config},
)
print("첫 번째 응답:", result1["messages"][-1].content)

# 이전 대화를 기억하는 두 번째 질문
result2 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "이전 결과에 10을 더하세요."}]},
    config={**config, **lf_config},
)
print("두 번째 응답:", result2["messages"][-1].content)

첫 번째 응답: 7 × 6 = 42입니다.


두 번째 응답: 이전 결과인 42에 10을 더하면 52입니다.


## 2.7 Tavily 검색 도구 연동 (선택)

웹 검색 도구를 추가하여 실제 정보를 검색합니다.

Tavily는 AI 에이전트를 위해 설계된 검색 API입니다. 

`TAVILY_API_KEY`가 설정된 경우에만 이 셀이 실행됩니다.

In [ ]:
# Tavily API 키가 있는 경우에만 실행
if os.environ.get("TAVILY_API_KEY"):
    from langchain_tavily import TavilySearch

    search_tool = TavilySearch(max_results=3)
    
    search_agent = create_agent(
        model=model,
        tools=[search_tool],
        system_prompt="당신은 리서치 어시스턴트입니다. 웹 검색을 통해 질문에 답하세요.",
    )
    
    result = search_agent.invoke(
        {"messages": [{"role": "user", "content": "LangChain의 최신 버전은 무엇인가요?"}]},
        config=lf_config,
    )
    print("검색 에이전트 응답:")
    print(result["messages"][-1].content)
else:
    print("\u26a0 TAVILY_API_KEY가 설정되지 않았습니다. 이 셀을 건너뜁니다.")

## 2.8 요약

이 노트북에서 다룬 내용:

| 주제 | 핵심 API | 설명 |
|------|----------|------|
| 도구 정의 | `@tool` | 함수에 데코레이터를 추가하여 에이전트 도구로 변환 |
| 에이전트 생성 | `create_agent()` | 모델 + 도구 + 시스템 프롬프트를 결합 |
| 동기 실행 | `agent.invoke()` | 완전한 응답을 한 번에 반환 |
| 스트리밍 실행 | `agent.stream()` | 각 단계의 업데이트를 실시간으로 반환 |
| 멀티턴 대화 | `InMemorySaver` + `thread_id` | 체크포인터로 대화 상태를 저장/복원 |
| 검색 도구 | `TavilySearch` | 웹 검색으로 실시간 정보에 접근 |

### 다음 단계

다음 노트북에서는 더 복잡한 에이전트 패턴을 살펴봅니다:
- 커스텀 시스템 프롬프트 설계
- 여러 도구를 조합한 에이전트
- 에러 처리와 폴백 전략